# External data collection- Dubrovnik Port data

This notebook retrieves historical ship arrivals and departures for Dubrovnik Port, from year 2024 to 2025.

Source: MyShipTracking.com, www.dubrovnikport.hr

Output: CSV file saved to 'uber_project/data/raw_data/ship_hourly_2024_2025'

# Initial approach: HTML table scraping
After investigation, discovered that the public webpage only provides current port calls, without historical filtering.
Therefore, switched to official MyShipTracking API to retrieve historical port call records.

### Setup and imports

In [32]:
import requests 
from bs4 import BeautifulSoup
import pandas as pd

### Configuration

In [33]:
url = 'https://www.myshiptracking.com/ports-arrivals-departures/?pid=2826'

response = requests.get(url)
response.status_code

200

In [34]:
html = response.text

### Parsing table using BeautifulSoup

In [35]:
soup = BeautifulSoup(html, 'html.parser')

In [36]:
table = soup.find('table')

In [22]:
headers = [th.get_text(strip=True) for th in table.find_all('th')]

In [23]:
rows = table.find_all('tr')

In [24]:
data = []

for row in rows:
    cols = row.find_all('td')
    cols = [col.text.strip() for col in cols]
    if cols:
        data.append(cols)

### Converting data to DataFrame

In [28]:
df = pd.DataFrame(data, columns=headers)

In [29]:
df.head()

,,Event,Time,Port,Vessel
0,,Arrival,2026-02-15 14:21,DUBROVNIK,AENONA [HR]
1,,Departure,2026-02-15 14:17,DUBROVNIK,HANIBAL LUCIC [HR]
2,,Arrival,2026-02-15 11:09,DUBROVNIK,STAR LEGEND [HR]
3,,Departure,2026-02-15 08:45,DUBROVNIK,PREMUDA [HR]


In [30]:
df.tail()


,,Event,Time,Port,Vessel
0,,Arrival,2026-02-15 14:21,DUBROVNIK,AENONA [HR]
1,,Departure,2026-02-15 14:17,DUBROVNIK,HANIBAL LUCIC [HR]
2,,Arrival,2026-02-15 11:09,DUBROVNIK,STAR LEGEND [HR]
3,,Departure,2026-02-15 08:45,DUBROVNIK,PREMUDA [HR]


# Approach 2 : Official API Port calls
Although the API provides historical port calls data, the free trial version is limited to:

-last 20 days only

-maximum 30 records per request

-credit-based system

### Decision: 
Ship traffic data will be excluded from the project unless an alternative website historical open dataset is found.

# Approach 3 : Official Dubrovnik Port documentation
Historical cruise ship PDF found at website www.portdubrovnik.hr , containing date and time of cruise ship arrivals and departures.

In [23]:
pdf_2024 = 'Download ship schedule - 2024.pdf'

In [24]:
pdf_2025 = 'Download schedule - ship arrivals 2025.pdf'

### Read PDF 

In [25]:
import pdfplumber
import pandas as pd

In [29]:
all_tables = []

with pdfplumber.open(pdf_2024) as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            df = pd.DataFrame(table)
            all_tables.append(df)
ships_2024 = pd.concat(all_tables, ignore_index = True)
ships_2024.head()


,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,CRUISE SHIP ARRIVALS TO DUBROVNIK,None,None,None,None,None,None,None,None,Berth,None,2024,None,NaN
1,Time of IMO number Ship's name Ship's flag Nam...,None,None,None,None,None,None,None,None,Gruž,Old\nTown,Len(m),GT,NaN
2,None,None,None,None,None,None,None,None,None,401,119,,None,NaN
3,None,None,None,None,None,None,None,None,None,6,0,None,None,NaN
4,03.01.2024,None,None,Wednesday,None,None,None,None,,1,0,,,NaN


In [30]:
ships_2024.tail()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
869,Calls,2024,None,None,None,None,None,None,None,None,None,None,None,Total
870,None,1,2,3,4,5,6,7,8,9,10,11,12,None
871,Luka Dubrovnik,6,5,10,20,44,49,53,54,68,65,19,8,401
872,Sidrište stari grad,0,0,0,2,8,20,30,28,22,8,1,0,119
873,Total,6,5,10,22,52,69,83,82,90,73,20,8,520


In [35]:
all_tables = []

with pdfplumber.open(pdf_2025) as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            df = pd.DataFrame(table)
            all_tables.append(df)
ships_2025 = pd.concat(all_tables, ignore_index = True)
ships_2025.head()
    

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,CRUISE SHIP ARRIVALS TO DUBROVNIK,None,None,None,None,None,None,None,None,Berth,None,2025,None,NaN
1,Time of IMO number Ship's name Ship's flag Nam...,None,None,None,None,None,None,None,None,Gruž,Old\nTown,Len(m),GT,NaN
2,None,None,None,None,None,None,None,None,None,409,3,,None,NaN
3,None,None,None,None,None,None,None,None,None,11,0,None,None,NaN
4,01.01.2025,None,None,Wednesday,None,None,None,None,,1,0,,,NaN


In [36]:
ships_2025.tail()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
757,Calls,2025,None,None,None,None,None,None,None,None,None,None,None,Total
758,None,1,2,3,4,5,6,7,8,9,10,11,12,None
759,Luka Dubrovnik,11,2,11,27,47,48,58,58,55,57,22,13,409
760,Sidrište stari grad,0,0,0,0,2,1,0,0,0,0,0,0,3
761,Total,11,2,11,27,49,49,58,58,55,57,22,13,412


### Join tables

In [38]:
ships_all = pd.concat([ships_2024, ships_2025], ignore_index = True)
ships_all.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,CRUISE SHIP ARRIVALS TO DUBROVNIK,None,None,None,None,None,None,None,None,Berth,None,2024,None,NaN
1,Time of IMO number Ship's name Ship's flag Nam...,None,None,None,None,None,None,None,None,Gruž,Old\nTown,Len(m),GT,NaN
2,None,None,None,None,None,None,None,None,None,401,119,,None,NaN
3,None,None,None,None,None,None,None,None,None,6,0,None,None,NaN
4,03.01.2024,None,None,Wednesday,None,None,None,None,,1,0,,,NaN


In [39]:
ships_all.shape

(1636, 14)

In [42]:
ships_all.to_csv('../data/raw_data/ship_hourly_2024_2025', index = False)